# Exact patient-level machine unlearning in ConMedRL

This self-contained example demonstrates the supported unlearning guarantee:

1. build a baseline offline-RL dataset from synthetic ICU source tables;
2. perform a short FQI/FQE connectivity run and bind its artifacts to the dataset hash;
3. withdraw one patient and rebuild every derived data artifact from source;
4. reject and invalidate the stale baseline models;
5. initialize and train fresh FQI/FQE models on retained data;
6. verify the new model manifest and inspect the withdrawal audit trail.

ConMedRL does **not** mutate old model weights and call that unlearning. The supported method is full retraining after exact data removal. The tiny training runs below verify integration only and are not clinical experiments.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from ConMedRL import (
    ModelArtifactManifest,
    ModelCompatibilityError,
    RLConfig_custom,
    RLTraining,
    TrainDataLoader,
    ValTestDataLoader,
    assert_model_compatible,
    exact_retrain,
)
from ConMedRL.conmedrl import save_ocrl_models_and_data
from ConMedRL.data import (
    PreprocessConfig,
    build_dataset,
    rebuild_dataset_after_withdrawal,
)

WORK_DIR = Path(tempfile.mkdtemp(prefix="conmedrl_unlearning_"))
SOURCE_DIR = WORK_DIR / "synthetic_sicdb"
SOURCE_DIR.mkdir()
print("Temporary demonstration directory:", WORK_DIR)

In [ ]:
def write_synthetic_sicdb(root, n_patients=12):
    """Create the smallest SICdb-like source needed for this demonstration."""
    pd.DataFrame([
        {
            "ReferenceGlobalID": 707,
            "ReferenceValue": "HeartRateECG",
            "ReferenceName": "SignalFloat",
            "ReferenceDescription": "",
            "ReferenceUnit": "/min",
            "LOINC_code": np.nan,
        },
        {
            "ReferenceGlobalID": 710,
            "ReferenceValue": "SPO2",
            "ReferenceName": "SignalFloat",
            "ReferenceDescription": "",
            "ReferenceUnit": "%",
            "LOINC_code": np.nan,
        },
        {
            "ReferenceGlobalID": 3123,
            "ReferenceValue": "RASS - Richmond Agitation-Sedation Scale",
            "ReferenceName": "Scores",
            "ReferenceDescription": "",
            "ReferenceUnit": np.nan,
            "LOINC_code": np.nan,
        },
    ]).to_csv(root / "d_references.csv", index=False)

    cases, signals = [], []
    for index in range(n_patients):
        case_id = 1000 + index
        cases.append({
            "CaseID": case_id,
            "PatientID": 2000 + index,
            "ICUOffset": 3600,
            "TimeOfStay": 24 * 3600,
            "DischargeState": 2202,
            "OffsetOfDeath": np.nan,
            "Sex": 735 if index % 2 else 736,
            "WeightOnAdmission": 60 + index,
            "HeightOnAdmission": 170,
            "AgeOnAdmission": 40 + index,
            "HospitalUnit": 3,
            "OffsetAfterFirstAdmission": index * 30 * 86400,
        })
        for offset, heart_rate in ((7200, 70 + index), (43200, 75 + index)):
            signals.extend([
                {"CaseID": case_id, "DataID": 707, "Offset": offset, "Val": heart_rate},
                {"CaseID": case_id, "DataID": 710, "Offset": offset, "Val": 96},
            ])

    pd.DataFrame(cases).to_csv(root / "cases.csv", index=False)
    pd.DataFrame(signals).to_csv(root / "data_float_h.csv", index=False)
    pd.DataFrame(
        columns=["CaseID", "LaboratoryID", "Offset", "LaboratoryValue"]
    ).to_csv(root / "laboratory.csv", index=False)


write_synthetic_sicdb(SOURCE_DIR)
print("Synthetic subjects:", list(range(2000, 2012)))

In [ ]:
baseline_config = PreprocessConfig(
    database="sicdb",
    task="discharge",
    data_dir=SOURCE_DIR,
    output_dir=WORK_DIR / "baseline_data",
    output_formats=("csv",),
    min_variable_coverage=0.01,
)
baseline = build_dataset(baseline_config)

baseline_subjects = set().union(*[
    set(split.outcome["subject_id"])
    for split in baseline.splits.values()
])
print("Baseline dataset hash:", baseline.content_hash)
print("Baseline subjects:", len(baseline_subjects))
print("State/action/constraints:", baseline.state_dim, baseline.action_dim, baseline.num_constraints)

In [ ]:
def train_fresh_fqi_fqe(bundle, output_dir, seed, training_method):
    """Initialize fresh agents and run one integration step, never reuse weights."""
    batch_size = min(4, len(bundle.train.outcome))
    config = RLConfig_custom(
        algo_name="FQI/FQE exact-retrain demonstration",
        gamma=0.99,
        batch_size=batch_size,
        train_eps=1,
        train_eps_steps=1,
        weight_decay_fqi=0.0,
        weight_decay_fqe=0.0,
        optim_fqi="torch.optim.Adam",
        optim_fqe="torch.optim.Adam",
        loss_fqi="nn.MSELoss()",
        loss_fqe="nn.MSELoss()",
        memory_capacity=len(bundle.train.outcome),
        target_update=1,
        tau=0.01,
        lr_fqi=1e-3,
        lr_fqe_obj=1e-3,
        lr_fqe_con_list=[1e-3] * bundle.num_constraints,
        lr_lambda_list=[0.0] * bundle.num_constraints,
        constraint_num=bundle.num_constraints,
        # Limits are unused because constraint=False below. This is not a clinical run.
        threshold_list=[0.0] * bundle.num_constraints,
        device_type="cpu",
        activation_function_fqi="relu",
        activation_params_fqi={},
        activation_function_fqe="relu",
        activation_params_fqe={},
        random_seed=seed,
    )

    train_loader = TrainDataLoader(config, **bundle.loader_kwargs("train"))
    train_loader.data_buffer_train(
        bundle.loader_action, done_condition=None,
        num_constraint=bundle.num_constraints,
    )
    val_loader = ValTestDataLoader(config, **bundle.loader_kwargs("val"))
    val_loader.data_buffer(
        bundle.loader_action, done_condition=None,
        num_constraint=bundle.num_constraints,
    )

    trainer = RLTraining(
        config, bundle.state_dim, bundle.action_dim,
        train_loader.data_torch_loader_train,
        val_loader.data_torch_loader,
    )
    fqi = trainer.fqi_agent_config(hidden_layers=[8], seed=seed)
    fqe_obj = trainer.fqe_agent_config(
        fqi, hidden_layers=[8], eval_target="obj", seed=seed + 1,
    )
    fqe_constraints = [
        trainer.fqe_agent_config(
            fqi, hidden_layers=[8], eval_target=index, seed=seed + 2 + index,
        )
        for index in range(bundle.num_constraints)
    ]
    trainer.train(
        fqi, fqe_obj, fqe_constraints,
        constraint=False, save_num=1, z_value=1.96,
    )

    saved = save_ocrl_models_and_data(
        fqi, fqe_obj, fqe_constraints, trainer,
        constraint_names=[f"constraint_{i}" for i in range(bundle.num_constraints)],
        model_save_path=str(Path(output_dir) / "models"),
        data_save_path=str(Path(output_dir) / "metrics"),
        save_date=False,
        dataset_bundle=bundle,
        seeds={"training": seed, "fqi": seed, "fqe_objective": seed + 1},
        training_method=training_method,
    )
    if saved is None or "manifest" not in saved:
        raise RuntimeError("Training artifacts were not saved with lineage.")
    return saved


baseline_artifacts = train_fresh_fqi_fqe(
    baseline, WORK_DIR / "baseline_training", seed=31, training_method="baseline"
)
old_model_manifest = baseline_artifacts["manifest"]
assert_model_compatible(old_model_manifest, baseline)
print("Baseline model is bound to:", ModelArtifactManifest.read(old_model_manifest).dataset_content_hash)

In [ ]:
import json

withdrawn_subject_ids = [2001]
retained = rebuild_dataset_after_withdrawal(
    prior_manifest_path=baseline.written_files["manifest"],
    source_config=baseline_config,
    withdrawn_subject_ids=withdrawn_subject_ids,
    output_dir=WORK_DIR / "retained_data",
    purge_prior_cache=True,
)

retained_subjects = set().union(*[
    set(split.outcome["subject_id"])
    for split in retained.splits.values()
])
assert 2001 in baseline_subjects
assert 2001 not in retained_subjects
assert retained.content_hash != baseline.content_hash

retained_manifest = json.loads(
    Path(retained.written_files["manifest"]).read_text(encoding="utf-8")
)
assert "withdrawn_subject_ids" not in retained_manifest["config"]
print("Retained dataset hash:", retained.content_hash)
print("Redacted request metadata:", {
    "withdrawal_count": retained.config.withdrawal_count,
    "withdrawal_digest": retained.config.withdrawal_digest,
})
print("Affected counts:", retained.report["lineage"]["affected_counts"])

In [ ]:
stale_model_blocked = False
try:
    assert_model_compatible(old_model_manifest, retained)
except ModelCompatibilityError as error:
    stale_model_blocked = True
    print("Expected stale-model rejection:", error)
assert stale_model_blocked


def retrain_callback(retained_bundle):
    return train_fresh_fqi_fqe(
        retained_bundle,
        WORK_DIR / "retained_training",
        seed=31,
        training_method="exact_retrain",
    )


unlearning_run = exact_retrain(
    retrain_callback=retrain_callback,
    bundle=retained,
    seed=31,
    request_digest=retained.config.withdrawal_digest,
    invalidated_manifest=old_model_manifest,
)
new_model_manifest = unlearning_run["result"]["manifest"]

assert unlearning_run["method"] == "exact_retrain"
assert ModelArtifactManifest.read(old_model_manifest).status == "invalidated"
assert_model_compatible(new_model_manifest, retained)
print("Old model status:", ModelArtifactManifest.read(old_model_manifest).status)
print("New model status:", ModelArtifactManifest.read(new_model_manifest).status)
print("Exact retrain dataset hash:", unlearning_run["dataset_content_hash"])

In [ ]:
audit_summary = {
    "method": unlearning_run["method"],
    "request_digest": retained.report["lineage"]["request_digest"],
    "parent_manifest_digest": retained.report["lineage"]["parent_manifest_digest"],
    "old_dataset_hash": baseline.content_hash,
    "retained_dataset_hash": retained.content_hash,
    "affected_counts": retained.report["lineage"]["affected_counts"],
    "old_dataset_status": json.loads(
        Path(baseline.written_files["manifest"]).read_text(encoding="utf-8")
    )["status"],
    "old_model_status": ModelArtifactManifest.read(old_model_manifest).status,
    "new_model_status": ModelArtifactManifest.read(new_model_manifest).status,
    "new_model_training_method": ModelArtifactManifest.read(
        new_model_manifest
    ).training_method,
}
audit_summary

## Operational requirements

- A withdrawal request is cumulative: later generations must again supply all previously withdrawn IDs because persisted manifests contain only a count and digest.
- Keep source data available for rebuilding. Editing processed CSV rows is not exact unlearning because the split, imputer, scaler, and costs would remain fitted on withdrawn data.
- Do not serve a model without `assert_model_compatible(model_manifest, current_bundle)`.
- Archive invalidated manifests for audit, but block their artifacts from inference.
- A production retraining callback must construct new model and optimizer objects and complete the intended FQI/FQE convergence and held-out evaluation protocol.
- Deterministic seeds improve repeatability; CUDA runs are not guaranteed to be bitwise identical across hardware or driver versions.
- This feature manages data and model lineage. It is not a legal certification of compliance and does not implement approximate gradient-based unlearning.